# Mixed-Context Answering with LlamaIndex
### Combining Structured Data Analysis and Text Retrieval in One QA System

**Project -- NLP / Retrieval and Agentic Workflows Portfolio**

Most real business questions are not purely tabular and not purely textual.
They mix both.

Examples:
- Which product line had the highest revenue decline, and what policy change explains it?
- Did refund rates rise after the warranty update in Europe?
- Which region is underperforming, and what operational notes mention likely causes?

A spreadsheet alone cannot answer these, and a document retriever alone cannot answer them either.
You need a **mixed-context system** that can reason over:

1. **Structured context**: tables, KPIs, group-bys, time windows, aggregates
2. **Unstructured context**: product notes, policy memos, support summaries, analyst reports

This notebook shows how LlamaIndex supports that pattern, and more importantly,
how **mixed-context answering** actually works under the hood.

## What You Will Learn

| Part | Topic | Why It Matters |
|---|---|---|
| 1 | Build a structured sales table | Numeric truth lives here |
| 2 | Build a document knowledge base | Explanations live here |
| 3 | Build separate engines | Tables and text require different retrieval logic |
| 4 | Route user questions | Decide which engine should answer which part |
| 5 | Fuse evidence | Final answer needs both signals |
| 6 | Compare failure modes | Mixed-context systems fail in specific ways |
| 7 | Map to real LlamaIndex APIs | Move from concept to production |

## Mixed-Context Answering: The Core Idea

A mixed-context question usually contains **two hidden subproblems**:

```
User question:
  'Why did laptop revenue fall in Europe after Q2, and what notes explain it?'

Subproblem A -- Structured analytics
  - What counts as 'fall'?
  - Which quarters?
  - How much did revenue change?

Subproblem B -- Text retrieval
  - Which memo or report mentions Europe + laptop + post-Q2 issues?
  - Is the explanation about warranty, supply chain, or discounts?

Final answer
  = numeric finding + textual evidence + synthesis
```

### Why ordinary RAG is not enough

Plain RAG is strong at retrieving explanatory text, but weak at exact aggregation.
If you ask it to compute revenue deltas from raw rows, it often hallucinates.

### Why SQL alone is not enough

A SQL or pandas engine can compute exact metrics, but cannot explain *why* a trend happened
unless that explanation is encoded in structured columns.

### The mixed-context solution

Use one engine for tables, one engine for documents, then route and fuse.

## Environment Setup

In [ ]:
import subprocess, sys
for pkg in ['pandas', 'numpy', 'matplotlib', 'seaborn', 'plotly']:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
print('Ready.')

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import math
import re
import hashlib
from collections import Counter
from dataclasses import dataclass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

pd.set_option('display.max_columns', 30)
pd.set_option('display.max_colwidth', 120)
plt.rcParams['figure.figsize'] = (14, 5)
sns.set_style('whitegrid')
np.random.seed(42)
print('Imports OK')

## Part 1: Structured Context -- Sales and Support Data

We will create a synthetic but realistic quarterly retail dataset.
This is the **structured truth layer** used for exact analysis.

In [ ]:
quarters = ['2025-Q1', '2025-Q2', '2025-Q3', '2025-Q4']
regions = ['North America', 'Europe', 'APAC']
products = ['Laptop Pro', 'Laptop Air', 'Phone Max', 'Tablet Plus']

base_revenue = {
    'Laptop Pro': 420000,
    'Laptop Air': 310000,
    'Phone Max': 520000,
    'Tablet Plus': 260000,
}

region_mult = {'North America': 1.18, 'Europe': 0.92, 'APAC': 0.98}
quarter_mult = {'2025-Q1': 1.00, '2025-Q2': 1.06, '2025-Q3': 0.95, '2025-Q4': 1.12}

rows = []
for q in quarters:
    for r in regions:
        for p in products:
            revenue = base_revenue[p] * region_mult[r] * quarter_mult[q]
            units = revenue / {'Laptop Pro': 1400, 'Laptop Air': 980, 'Phone Max': 880, 'Tablet Plus': 620}[p]
            refunds = revenue * {'Laptop Pro': 0.060, 'Laptop Air': 0.045, 'Phone Max': 0.038, 'Tablet Plus': 0.042}[p]
            support = units * {'Laptop Pro': 0.17, 'Laptop Air': 0.11, 'Phone Max': 0.07, 'Tablet Plus': 0.08}[p]

            if q in ['2025-Q3', '2025-Q4'] and r == 'Europe' and p == 'Laptop Pro':
                revenue *= 0.78
                refunds *= 1.45
                support *= 1.55
            if q in ['2025-Q3', '2025-Q4'] and r == 'Europe' and p == 'Laptop Air':
                revenue *= 0.87
                refunds *= 1.32
                support *= 1.28
            if q == '2025-Q4' and r == 'APAC' and p == 'Phone Max':
                revenue *= 1.13
                units *= 1.10
            if q in ['2025-Q2', '2025-Q3'] and r == 'North America' and p == 'Tablet Plus':
                revenue *= 0.91
                support *= 1.18

            rows.append({
                'quarter': q,
                'region': r,
                'product': p,
                'revenue': round(revenue, 2),
                'units_sold': int(round(units)),
                'refund_cost': round(refunds, 2),
                'support_tickets': int(round(support)),
                'avg_discount_pct': round(np.random.uniform(4.0, 11.0), 2),
            })

sales_df = pd.DataFrame(rows)
sales_df['refund_rate'] = sales_df['refund_cost'] / sales_df['revenue']
sales_df['support_rate'] = sales_df['support_tickets'] / sales_df['units_sold']

print(sales_df.shape)
sales_df.head(8)

In [ ]:
summary = sales_df.groupby(['quarter', 'region'])[['revenue', 'refund_cost', 'support_tickets']].sum().reset_index()
summary

### What the table can answer well

The structured layer is excellent for:
- exact sums, averages, and deltas
- quarter-over-quarter comparisons
- product or region ranking
- refund and support rate analysis

It is not good at answering *why* a trend happened if that explanation appears only in documents.

## Part 2: Unstructured Context -- Operational Notes and Policy Memos

Next we build a small document collection. This is the **explanation layer**.
Each document contains business context that is not available in the table.

In [ ]:
@dataclass
class Document:
    text: str
    title: str
    doc_type: str
    quarter: str
    region: str
    product: str

    def __post_init__(self):
        self.id = hashlib.sha256((self.title + self.text).encode()).hexdigest()[:10]


docs = [
    Document(
        title='EU Laptop Warranty Memo',
        doc_type='policy_memo',
        quarter='2025-Q3',
        region='Europe',
        product='Laptop Pro',
        text='Beginning in 2025-Q3, the Europe region adopted a stricter warranty intake policy for Laptop Pro and Laptop Air. Support teams were instructed to accept more early-hardware complaints rather than redirect customers to self-service diagnostics. This change improved customer satisfaction but increased support tickets and refund handling volumes.'
    ),
    Document(
        title='EU Keyboard Quality Incident',
        doc_type='quality_report',
        quarter='2025-Q3',
        region='Europe',
        product='Laptop Pro',
        text='A supplier batch affected keyboard assemblies for Laptop Pro units shipped into Europe during late Q2 and early Q3. Customers reported intermittent key failures, causing elevated returns and negative reseller sentiment. Sales teams noted that enterprise buyers delayed refresh cycles until the revised batch arrived.'
    ),
    Document(
        title='EU Laptop Air Margin Review',
        doc_type='analyst_note',
        quarter='2025-Q4',
        region='Europe',
        product='Laptop Air',
        text='Laptop Air revenue in Europe softened after Q2 because channel partners reduced shelf space while support queues lengthened. The issue was not a collapse in demand alone; slower fulfilment and warranty processing reduced conversion on repeat purchases and student bundles.'
    ),
    Document(
        title='APAC Phone Max Holiday Campaign',
        doc_type='marketing_report',
        quarter='2025-Q4',
        region='APAC',
        product='Phone Max',
        text='APAC Phone Max demand rose sharply in Q4 after the holiday bundle campaign paired the device with a premium accessory offer. Retail partners reported improved conversion and low return pressure, making the campaign one of the strongest quarter-end programs.'
    ),
    Document(
        title='North America Tablet Bundle Note',
        doc_type='analyst_note',
        quarter='2025-Q2',
        region='North America',
        product='Tablet Plus',
        text='Tablet Plus underperformed in North America during Q2 and Q3 after a bundle promotion was removed from education channels. Traffic remained healthy, but the average order value and conversion rate weakened without the classroom accessory pack.'
    ),
    Document(
        title='Returns Operations Handbook Update',
        doc_type='operations_policy',
        quarter='2025-Q3',
        region='Europe',
        product='Laptop Pro',
        text='The returns operations handbook was updated in Q3 to encourage regional triage teams in Europe to authorise more direct replacements for premium laptop complaints. This reduced escalation loops but increased immediate refund and replacement accounting within the quarter.'
    ),
    Document(
        title='Global Customer Service Summary',
        doc_type='support_summary',
        quarter='2025-Q4',
        region='Global',
        product='All',
        text='Across all regions, customer service pressure was highest for Europe laptops, where warranty-policy changes and component quality issues overlapped. APAC mobile support remained relatively stable despite strong Q4 sales growth.'
    ),
    Document(
        title='Q4 Executive Commentary',
        doc_type='executive_note',
        quarter='2025-Q4',
        region='Europe',
        product='Laptop Pro',
        text='Europe laptop weakness after Q2 should be interpreted as a quality-and-operations problem, not as broad brand deterioration. The main drivers were a faulty supplier batch, slower replacement throughput, and a temporary shift in reseller confidence.'
    ),
    Document(
        title='Discount Strategy Brief',
        doc_type='pricing_note',
        quarter='2025-Q4',
        region='Europe',
        product='Laptop Air',
        text='Discounting on Laptop Air in Europe was intentionally restrained in Q4 to protect margin after refund costs rose. This limited the ability to offset support-related conversion headwinds with aggressive promotions.'
    ),
    Document(
        title='Channel Inventory Pulse',
        doc_type='channel_report',
        quarter='2025-Q3',
        region='Europe',
        product='Laptop Pro',
        text='Several European channel partners reduced forward inventory purchases for Laptop Pro after support complaints increased. Inventory caution was temporary but materially affected Q3 sell-in volumes.'
    ),
    Document(
        title='Support Automation Rollout',
        doc_type='ops_update',
        quarter='2025-Q4',
        region='North America',
        product='All',
        text='North America support teams reduced average handling time in Q4 through a triage automation rollout. Ticket growth did not translate into proportional service costs because routing became more efficient.'
    ),
    Document(
        title='Cross-Region Product Health Summary',
        doc_type='health_report',
        quarter='2025-Q4',
        region='Global',
        product='All',
        text='The strongest mixed signal in 2025 was Europe laptops: revenue fell while support demand and refund costs rose. That pattern usually indicates operational or quality issues rather than simple price elasticity.'
    ),
]

docs_df = pd.DataFrame([d.__dict__ for d in docs])
docs_df[['title', 'doc_type', 'quarter', 'region', 'product']].head(12)

### What the document layer can answer well

The text layer is strong for:
- explanations and causal narratives
- policy changes
- operational incidents
- support or channel commentary

It is weak for exact aggregation, ranking, and numerical truth.

## Part 3: Build Separate Engines

A mixed-context system works best when each modality gets its own engine.

- **Structured engine**: computes exact metrics from the table
- **Text engine**: retrieves semantically relevant notes from documents

This is the conceptual shape of a LlamaIndex setup using `PandasQueryEngine` plus `VectorStoreIndex`.

In [ ]:
class TinyEmbedder:
    def __init__(self, dim=96):
        self.dim = dim
        self.vocab = {}
        self.idf = {}
        self.proj = None

    def fit(self, texts):
        df = Counter()
        for text in texts:
            for tok in set(self._tok(text)):
                df[tok] += 1
        self.vocab = {t: i for i, t in enumerate(sorted(df))}
        n = len(texts)
        self.idf = {t: math.log(n / (1 + c)) for t, c in df.items()}
        np.random.seed(42)
        self.proj = np.random.randn(len(self.vocab), self.dim) / math.sqrt(self.dim)

    def _tok(self, text):
        return re.findall(r'[a-z0-9]+', text.lower())

    def encode(self, text):
        vec = np.zeros(len(self.vocab))
        tf = Counter(self._tok(text))
        for tok, cnt in tf.items():
            if tok in self.vocab:
                vec[self.vocab[tok]] = cnt * self.idf.get(tok, 1.0)
        proj = vec @ self.proj
        norm = np.linalg.norm(proj)
        return proj / norm if norm > 0 else proj


embedder = TinyEmbedder(dim=96)
embedder.fit([d.text for d in docs])
doc_embeddings = {d.id: embedder.encode(d.text) for d in docs}
print('Embedder ready for text retrieval')

In [ ]:
class StructuredQueryEngine:
    def __init__(self, df):
        self.df = df.copy()

    def query(self, question):
        q = question.lower()
        df = self.df.copy()

        for region in ['Europe', 'North America', 'APAC']:
            if region.lower() in q:
                df = df[df['region'] == region]
        for product in ['Laptop Pro', 'Laptop Air', 'Phone Max', 'Tablet Plus']:
            if product.lower() in q:
                df = df[df['product'] == product]

        metric = 'revenue'
        if 'refund' in q:
            metric = 'refund_rate' if 'rate' in q else 'refund_cost'
        elif 'support' in q or 'ticket' in q:
            metric = 'support_rate' if 'rate' in q else 'support_tickets'
        elif 'unit' in q:
            metric = 'units_sold'
        elif 'discount' in q:
            metric = 'avg_discount_pct'

        if any(word in q for word in ['highest', 'top', 'largest']):
            grp_col = 'product' if 'product' in q or 'line' in q else 'region'
            tmp = df.groupby(grp_col)[metric].sum().sort_values(ascending=False).reset_index()
            top = tmp.iloc[0]
            return {
                'mode': 'structured',
                'metric': metric,
                'result_type': 'ranking',
                'data': tmp,
                'answer': f'{top[grp_col]} has the highest {metric} at {top[metric]:,.2f}.',
            }

        if any(word in q for word in ['fall', 'decline', 'drop', 'after q2']):
            pre = df[df['quarter'].isin(['2025-Q1', '2025-Q2'])][metric].sum()
            post = df[df['quarter'].isin(['2025-Q3', '2025-Q4'])][metric].sum()
            delta = post - pre
            pct = (delta / pre) if pre else 0.0
            return {
                'mode': 'structured',
                'metric': metric,
                'result_type': 'delta',
                'data': {'pre_q2': pre, 'post_q2': post, 'delta': delta, 'pct_change': pct},
                'answer': f'{metric} changed from {pre:,.2f} before/through Q2 to {post:,.2f} after Q2, a {pct:+.1%} change.',
            }

        total = df[metric].sum() if metric not in ['refund_rate', 'support_rate', 'avg_discount_pct'] else df[metric].mean()
        return {
            'mode': 'structured',
            'metric': metric,
            'result_type': 'aggregate',
            'data': total,
            'answer': f'{metric} is {total:,.2f}.',
        }


class TextRetrievalEngine:
    def __init__(self, documents, embeddings, embedder):
        self.documents = documents
        self.embeddings = embeddings
        self.embedder = embedder

    def query(self, question, top_k=3):
        q_emb = self.embedder.encode(question)
        scored = []
        for d in self.documents:
            score = float(self.embeddings[d.id] @ q_emb)
            scored.append((score, d))
        scored.sort(key=lambda x: -x[0])
        best = scored[:top_k]
        return {
            'mode': 'text',
            'hits': [
                {
                    'title': d.title,
                    'region': d.region,
                    'product': d.product,
                    'score': round(score, 4),
                    'text': d.text,
                }
                for score, d in best
            ],
            'answer': ' | '.join([f'{d.title}: {d.text[:110]}...' for score, d in best]),
        }


structured_engine = StructuredQueryEngine(sales_df)
text_engine = TextRetrievalEngine(docs, doc_embeddings, embedder)
print('Structured and text engines ready')

## Part 4: Mixed-Context Routing

A router decides whether a question needs:
- structured analysis only
- text retrieval only
- both

In LlamaIndex, this is commonly done with a router or sub-question query engine.

In [ ]:
class MixedContextRouter:
    STRUCTURED_HINTS = ['revenue', 'refund', 'support', 'rate', 'highest', 'lowest',
                        'top', 'average', 'sum', 'decline', 'fall', 'after q2', 'quarter']
    TEXT_HINTS = ['why', 'explain', 'note', 'memo', 'report', 'policy', 'cause',
                  'caused', 'reason', 'operational', 'quality', 'commentary']

    def decide(self, question):
        q = question.lower()
        needs_structured = any(h in q for h in self.STRUCTURED_HINTS)
        needs_text = any(h in q for h in self.TEXT_HINTS)
        if needs_structured and needs_text:
            return 'mixed'
        if needs_structured:
            return 'structured'
        if needs_text:
            return 'text'
        return 'mixed'


class MixedContextAnswerEngine:
    def __init__(self, structured_engine, text_engine, router):
        self.structured_engine = structured_engine
        self.text_engine = text_engine
        self.router = router

    def answer(self, question):
        route = self.router.decide(question)
        if route == 'structured':
            s = self.structured_engine.query(question)
            return {'route': route, 'structured': s, 'text': None, 'final': s['answer']}
        if route == 'text':
            t = self.text_engine.query(question)
            top = t['hits'][0]
            final = f"Top explanatory note: {top['title']} -- {top['text']}"
            return {'route': route, 'structured': None, 'text': t, 'final': final}

        s = self.structured_engine.query(question)
        t = self.text_engine.query(question)
        top = t['hits'][0]
        final = (
            f"Structured finding: {s['answer']} "
            f"Relevant note: {top['title']} explains that {top['text']}"
        )
        return {'route': route, 'structured': s, 'text': t, 'final': final}


router = MixedContextRouter()
mixed_engine = MixedContextAnswerEngine(structured_engine, text_engine, router)
print('Mixed-context router ready')

### How mixed-context answering works

The final answer engine does **three things**:

1. **Decompose intent**
   It checks whether the question asks for numeric analysis, explanation, or both.

2. **Call the right tools**
   The structured engine computes exact numbers. The text engine retrieves explanatory notes.

3. **Fuse evidence**
   The final response combines exact measurements with cited context.

This fusion step is the difference between:
- a table query engine
- a document retriever
- a true mixed-context QA system

## Part 5: Example Questions

In [ ]:
example_questions = [
    'Which product line had the largest revenue decline after Q2 in Europe?',
    'Why did Laptop Pro revenue fall in Europe after Q2?',
    'Did refund rates rise for Europe laptops, and what note explains it?',
    'What explains APAC Phone Max growth in Q4?',
    'What is the average support rate for Tablet Plus in North America?',
]

for q in example_questions:
    result = mixed_engine.answer(q)
    print('=' * 110)
    print('QUESTION:', q)
    print('ROUTE   :', result['route'])
    print('ANSWER  :', result['final'])

In [ ]:
demo_q = 'Why did Laptop Pro revenue fall in Europe after Q2?'
demo_result = mixed_engine.answer(demo_q)

print('Route:', demo_result['route'])
print('\nStructured evidence:')
print(demo_result['structured'])
print('\nTop text hits:')
pd.DataFrame(demo_result['text']['hits'])[['title', 'region', 'product', 'score']]

## Part 6: Evaluate the System

We evaluate three variants:

1. **Structured-only**: ignores documents
2. **Text-only**: ignores tables
3. **Mixed-context**: routes and fuses both

This makes the value of mixed-context answering concrete.

In [ ]:
EVAL_SET = [
    {
        'question': 'Why did Laptop Pro revenue fall in Europe after Q2?',
        'needs': 'mixed',
        'expected_doc': 'EU Keyboard Quality Incident',
        'expected_phrase': 'after Q2',
    },
    {
        'question': 'Did refund rates rise for Europe laptops, and what note explains it?',
        'needs': 'mixed',
        'expected_doc': 'EU Laptop Warranty Memo',
        'expected_phrase': 'refund',
    },
    {
        'question': 'What explains APAC Phone Max growth in Q4?',
        'needs': 'mixed',
        'expected_doc': 'APAC Phone Max Holiday Campaign',
        'expected_phrase': 'Q4',
    },
    {
        'question': 'Which product line had the highest revenue?',
        'needs': 'structured',
        'expected_doc': None,
        'expected_phrase': 'highest',
    },
    {
        'question': 'What note mentions Europe laptops and quality issues?',
        'needs': 'text',
        'expected_doc': 'Cross-Region Product Health Summary',
        'expected_phrase': 'quality',
    },
]

def run_eval(mode):
    rows = []
    for item in EVAL_SET:
        q = item['question']
        if mode == 'structured_only':
            out = {'route': 'structured', 'structured': structured_engine.query(q), 'text': None}
            final = out['structured']['answer']
        elif mode == 'text_only':
            out = {'route': 'text', 'structured': None, 'text': text_engine.query(q)}
            final = out['text']['answer']
        else:
            out = mixed_engine.answer(q)
            final = out['final']

        route_ok = 1.0 if (mode == 'mixed' and out['route'] == item['needs']) or mode != 'mixed' else 0.0
        doc_ok = 0.0
        if item['expected_doc'] is None:
            doc_ok = 1.0
        elif out.get('text') and out['text']['hits']:
            doc_ok = 1.0 if out['text']['hits'][0]['title'] == item['expected_doc'] else 0.0

        phrase_ok = 1.0 if item['expected_phrase'].lower() in final.lower() else 0.0
        composite = (route_ok + doc_ok + phrase_ok) / 3
        rows.append({
            'mode': mode,
            'question': q[:65],
            'route_ok': route_ok,
            'doc_ok': doc_ok,
            'phrase_ok': phrase_ok,
            'composite': composite,
        })
    return pd.DataFrame(rows)

eval_struct = run_eval('structured_only')
eval_text = run_eval('text_only')
eval_mixed = run_eval('mixed')

scores = pd.concat([eval_struct, eval_text, eval_mixed], ignore_index=True)
scores.groupby('mode')[['route_ok', 'doc_ok', 'phrase_ok', 'composite']].mean().round(4)

In [ ]:
score_summary = scores.groupby('mode')[['route_ok', 'doc_ok', 'phrase_ok', 'composite']].mean().reset_index()
score_summary

### Expected result

- Structured-only should do well on exact metrics but poorly on explanatory evidence
- Text-only should do better on explanation but poorly on exact numeric phrasing
- Mixed-context should dominate on the combined score

## Part 7: Visualisations

In [ ]:
fig = px.bar(
    score_summary.melt(id_vars='mode', var_name='metric', value_name='score'),
    x='metric', y='score', color='mode', barmode='group',
    title='Evaluation Metrics by System Variant',
    template='plotly_white', height=420
)
fig.update_yaxes(range=[0, 1])
fig.show()

In [ ]:
rev_by_region = sales_df.groupby(['quarter', 'region'])['revenue'].sum().reset_index()
fig = px.line(rev_by_region, x='quarter', y='revenue', color='region', markers=True,
              title='Revenue by Region Across Quarters', template='plotly_white', height=420)
fig.show()

In [ ]:
eu_laptops = sales_df[(sales_df['region'] == 'Europe') & (sales_df['product'].isin(['Laptop Pro', 'Laptop Air']))]
tmp = eu_laptops.groupby(['quarter', 'product'])[['revenue', 'refund_rate', 'support_rate']].mean().reset_index()
fig = make_subplots(rows=1, cols=3, subplot_titles=['Revenue', 'Refund Rate', 'Support Rate'])
for product in ['Laptop Pro', 'Laptop Air']:
    grp = tmp[tmp['product'] == product]
    fig.add_trace(go.Scatter(x=grp['quarter'], y=grp['revenue'], mode='lines+markers', name=product), row=1, col=1)
    fig.add_trace(go.Scatter(x=grp['quarter'], y=grp['refund_rate'], mode='lines+markers', name=product, showlegend=False), row=1, col=2)
    fig.add_trace(go.Scatter(x=grp['quarter'], y=grp['support_rate'], mode='lines+markers', name=product, showlegend=False), row=1, col=3)
fig.update_layout(template='plotly_white', height=420, title='Europe Laptop Health Signals')
fig.show()

In [ ]:
doc_counts = docs_df.groupby(['doc_type']).size().sort_values(ascending=False).reset_index(name='count')
fig = px.bar(doc_counts, x='doc_type', y='count', color='count',
             title='Document Collection Composition', template='plotly_white', height=380)
fig.show()

In [ ]:
retrieved_titles = []
for item in EVAL_SET:
    out = mixed_engine.answer(item['question'])
    if out.get('text') and out['text']['hits']:
        retrieved_titles.extend([h['title'] for h in out['text']['hits']])
hit_df = pd.DataFrame(Counter(retrieved_titles).items(), columns=['title', 'count']).sort_values('count', ascending=False)
fig = px.bar(hit_df, x='count', y='title', orientation='h',
             title='Most Frequently Retrieved Notes', template='plotly_white', height=420)
fig.show()

In [ ]:
route_examples = []
for item in EVAL_SET:
    out = mixed_engine.answer(item['question'])
    route_examples.append({'question': item['question'][:45] + '...', 'route': out['route']})
route_df = pd.DataFrame(route_examples)
fig = px.pie(route_df, names='route', title='Routes Chosen by Mixed-Context Router',
             template='plotly_white', height=380)
fig.show()

## Part 8: Failure Modes in Mixed-Context QA

Mixed-context systems fail in ways that single-modality systems do not.

| Failure Mode | What Happens | Example |
|---|---|---|
| Numeric hallucination | Text engine guesses numbers | 'Revenue fell by 40%' without computation |
| Explanation gap | Table engine gives exact metric but no cause | 'Refund rate rose' with no memo |
| Routing error | Router sends a mixed question to only one engine | loses one modality |
| Evidence mismatch | Correct metric + irrelevant note | false explanation |
| Over-fusion | Too many retrieved notes muddy the answer | synthesis becomes vague |

In [ ]:
failure_demo_questions = [
    'Why did refund costs rise in Europe laptops after Q2?',
    'Which region had the highest revenue and what memo explains it?',
    'What is the exact support rate for Laptop Pro in Europe and why?',
]

rows = []
for q in failure_demo_questions:
    s = structured_engine.query(q)
    t = text_engine.query(q)
    m = mixed_engine.answer(q)
    rows.append({
        'question': q,
        'structured_only': s['answer'],
        'text_only': t['answer'][:120] + '...',
        'mixed': m['final'][:150] + '...',
    })
pd.DataFrame(rows)

## Part 9: Mapping This to Real LlamaIndex APIs

The notebook so far used lightweight local components so the logic is transparent.
Here is the corresponding LlamaIndex architecture.

### Real LlamaIndex Version

```python
from llama_index.core import VectorStoreIndex, Settings
from llama_index.core.query_engine import RouterQueryEngine, SubQuestionQueryEngine
from llama_index.core.tools import QueryEngineTool
from llama_index.experimental.query_engine import PandasQueryEngine
from llama_index.core.schema import Document
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

Settings.llm = Ollama(model='qwen3', request_timeout=120.0)
Settings.embed_model = HuggingFaceEmbedding(model_name='BAAI/bge-small-en-v1.5')

pandas_engine = PandasQueryEngine(df=sales_df)
documents = [Document(text=d.text, metadata={
    'title': d.title, 'region': d.region, 'product': d.product
}) for d in docs]
text_index = VectorStoreIndex.from_documents(documents)
text_engine = text_index.as_query_engine(similarity_top_k=3)

structured_tool = QueryEngineTool.from_defaults(
    query_engine=pandas_engine,
    name='structured_sales_engine',
    description='Use for exact numeric analysis over revenue, refunds, support, and trends.'
)

text_tool = QueryEngineTool.from_defaults(
    query_engine=text_engine,
    name='ops_notes_engine',
    description='Use for policy memos, incidents, analyst commentary, and explanations.'
)

router_engine = RouterQueryEngine.from_defaults(
    query_engine_tools=[structured_tool, text_tool]
)

mixed_engine = SubQuestionQueryEngine.from_defaults(
    query_engine_tools=[structured_tool, text_tool]
)

response = mixed_engine.query(
    'Why did Laptop Pro revenue fall in Europe after Q2, and what notes explain it?'
)
print(response)
```

### What LlamaIndex adds in production

Compared with our transparent teaching implementation, LlamaIndex gives you:
- tool descriptions for routing
- stronger document indexing
- built-in retrievers and query engines
- sub-question decomposition
- response synthesis across tools
- easier swap-in of local or hosted LLMs

## Part 10: Production Design Guidelines

If you deploy a real mixed-context system, follow these rules:

1. **Never let the text retriever invent numeric truth**
   Numeric answers should come from structured engines, not narrative docs.

2. **Keep tool descriptions sharp**
   Router quality depends heavily on describing when each tool should be used.

3. **Fuse with evidence, not just prose**
   Final answers should cite both the metric and the retrieved note title.

4. **Prefer decomposition for mixed questions**
   One agentic pass often works worse than explicit sub-questioning.

5. **Measure routing separately from answer quality**
   A good retriever cannot fix a bad router.

## Summary

### The mixed-context pattern

```
User question
   -> detect whether it needs numbers, text, or both
   -> send numeric parts to structured engine
   -> send explanatory parts to text retriever
   -> fuse exact metrics with retrieved evidence
   -> return one grounded answer
```

### Key takeaways

| Insight | Why It Matters |
|---|---|
| Tables and docs answer different kinds of truth | exact vs explanatory |
| Mixed-context QA is a routing problem first | wrong routing breaks everything |
| Final answers must fuse both evidence types | one modality is not enough |
| LlamaIndex provides the right abstractions | query engines, tools, routers, sub-questions |
| Evaluation should test routing, retrieval, and synthesis separately | avoids hidden failure modes |

---
*Educational notebook -- NLP / LlamaIndex Mixed Context QA Portfolio.*